# Experiment 3 - perceptual after-effect

Edit `STUDY_NR = "sxxxxxx"` to your student ID.

Place the VS Code window on the right side of your screen (not fullscreen), so the pop-up window has room on the left and you can type your ratings without moving windows around.

Then press "Run All" in VS Code.

In [28]:
import csv
import random
import time
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

# The test face is only allowed to be on the screen for 0.5–1.0 seconds, and the inline notebook backend cannot do that - it just dumps the image into the output and leaves it there.
# So we use matplotlib for a real pop-up window instead. The first backend in this list that works on the machine is the one we get (macosx on a Mac, the other two on Windows/Linux).
for backend in ("macosx", "qtagg", "tkagg"):
    try:
        matplotlib.use(backend, force=True)
        break
    except Exception:
        continue

print("Showing images with backend:", matplotlib.get_backend())

Showing images with backend: macosx


In [ ]:
STUDY_NR = "sxxxxxx"  # --> **change this to the student ID** <--

SYNTH_DIR = Path("synthetic")      # the 11 synthetic faces we made in step 5/6
STIM_DIR = Path("synthetic_exp3")   # the same faces, but with a fixation point drawn on them

# The adapting stimuli are the two ends of our rating continuum - the most "male" and the most "female" face the model can produce. Staring at one of them is what makes the after-effect happen.
ADAPTORS = ["face_00.5.png", "face_05.5.png"]

# The test stimuli sit close to the neutral face. They have to be near the middle of the scale, because a face that already looks obviously male or female cannot shift much in either direction.
TESTS = ["face_02.5.png", "face_03.0.png", "face_03.5.png"]

REPEATS = 5  # how many times each test face is rated inside one adaptation condition

# 2 adaptors x 3 test faces = the six experimental conditions the assignment asks for.

INITIAL_ADAPT = 25.0        # seconds of adaptation at the start of a block - the assignment asks for 20-30 s
TOPUP_ADAPT = 20.0          # re-adaptation before every single trial, so the effect does not fade
TEST_DURATION = 0.75        # seconds the test face is visible - the notes ask for 0.5-1.0 s
BLANK_AFTER_TEST = 0.05     # seconds of grey screen after the test face - a picture stays on screen until something else is drawn on top of it, so we have to switch to grey to make the test face go away again

print(f"{len(ADAPTORS) * len(TESTS)} conditions, {len(TESTS) * REPEATS} trials per adaptation block")
print(f"Rough running time: {2 * (INITIAL_ADAPT + len(TESTS) * REPEATS * (TOPUP_ADAPT + 4)) / 60:.0f} minutes")

6 conditions, 15 trials per adaptation block
Rough running time: 13 minutes


In [30]:
# The assignment requires a fixation point in the centre of every image.


def draw_cross(img, value, half_len, half_thick):
    """Paint a plus-shaped mark of the given brightness in the middle of the image."""
    cy, cx = img.shape[0] // 2, img.shape[1] // 2
    img[cy - half_thick : cy + half_thick + 1, cx - half_len : cx + half_len + 1] = value
    img[cy - half_len : cy + half_len + 1, cx - half_thick : cx + half_thick + 1] = value


def add_fixation(img):
    """Return a copy of the face with a white cross on a black outline in the centre."""
    out = img.copy()
    draw_cross(out, 0, half_len=8, half_thick=2)    # dark outline first, so the cross is visible
    draw_cross(out, 255, half_len=6, half_thick=0)  # bright cross on top, also on a dark face
    return out


STIM_DIR.mkdir(exist_ok=True)

for name in ADAPTORS + TESTS:
    face = plt.imread(SYNTH_DIR / name)
    if face.ndim == 3:                      # some png files are saved as RGB even when they look grey - take one channel
        face = face[:, :, 0]
    if face.max() <= 1.0:                   # matplotlib reads png files as 0-1 floats, we want plain 0-255 grey values
        face = face * 255
    face = face.astype(float)

    plt.imsave(STIM_DIR / name, add_fixation(face), cmap="gray", vmin=0, vmax=255)

print(f"Wrote {len(ADAPTORS) + len(TESTS)} stimuli with a fixation point to {STIM_DIR}")

Wrote 5 stimuli with a fixation point to synthetic_exp3


In [31]:
# Everything below draws into one single window that stays open for the whole experiment. We swap the picture inside it instead of opening and closing windows,
# because a window that pops up and disappears would make the screen flash and steal the observer's attention.

stimuli = {name: plt.imread(STIM_DIR / name) for name in ADAPTORS + TESTS}
blank = np.full_like(np.asarray(stimuli[TESTS[0]], dtype=float), 0.5)
if blank.ndim == 3:
    blank[:, :, 3] = 1.0
draw_cross(blank, 0.0, half_len=8, half_thick=2)                            # the fixation point stays put between trials too
draw_cross(blank, 1.0, half_len=6, half_thick=0)


def open_screen():
    """Open the window we present everything in and return it."""
    fig, ax = plt.subplots(figsize=(6, 6))
    fig.canvas.manager.set_window_title("Experiment 3")
    handle = ax.imshow(blank, cmap="gray", vmin=0, vmax=1)
    ax.axis("off")
    fig.tight_layout(pad=0)
    fig.show()
    return fig, handle


def show(handle, image, seconds):
    """Put an image on the screen and leave it there for a set number of seconds."""
    handle.set_data(image)
    handle.set_clim(0, 1 if np.max(image) <= 1.0 else 255)
    plt.pause(0.001)  # forces the window to actually redraw before we start counting time
    deadline = time.perf_counter() + seconds
    while time.perf_counter() < deadline:
        # Small steps rather than one long sleep, so the window keeps responding and the timing does not drift. This matters most for the 0.75 s test presentation.
        plt.pause(min(0.02, max(0.001, deadline - time.perf_counter())))


def ask_rating():
    """Ask for a 1-5 rating in the notebook. Returns None if the participant wants to stop."""
    while True:
        answer = input("Rating 1(male)-5(female) (q to quit): ").strip()
        if answer == "q":
            return None
        if answer in "12345" and answer:
            return int(answer)
        print("Please type a number from 1 to 5, or q to stop.")

In [32]:
print("INSTRUCTIONS FOR THE PARTICIPANT")
print("- Keep your eyes on the cross in the middle of the image at all times.")
print("- A face will be shown for a long time. Keep looking at the cross, do not look around it.")
print("- Then a face flashes up very briefly. Rate that face, 1 = male and 5 = female.")
print("- Answer with your first impression. It is supposed to be hard.")
print("- Type your answer in the notebook, then look back at the cross straight away.\n")
input("Press Enter when the participant is ready and sitting comfortably...")

# One row of the results is one condition: a test face rated under one of the two adaptors.
results = {(adaptor, test): [] for adaptor in ADAPTORS for test in TESTS}

block_order = ADAPTORS.copy()
random.shuffle(block_order)

screen = None
stopped = False

try:
    fig, handle = open_screen()
    screen = fig

    for block_nr, adaptor in enumerate(block_order, start=1):
        print(f"\n--- Block {block_nr} of 2 ---")
        input("Press Enter to start the block, then look at the cross and do not look away...")

        # Long adaptation once at the start of the block. This is what tires out the neurons that code this end of the male-female dimension.
        show(handle, stimuli[adaptor], INITIAL_ADAPT)

        # Every test face appears REPEATS times, and the whole list is shuffled
        trials = TESTS * REPEATS
        random.shuffle(trials)

        for trial_nr, test in enumerate(trials, start=1):
            show(handle, stimuli[adaptor], TOPUP_ADAPT)     # top up the adaptation before each trial
            show(handle, stimuli[test], TEST_DURATION)      # the brief glimpse that is actually rated
            show(handle, blank, BLANK_AFTER_TEST)           # back to grey, so the face cannot be studied

            rating = ask_rating()
            if rating is None:
                stopped = True
                break

            results[(adaptor, test)].append(rating)
            print(f"  trial {trial_nr}/{len(trials)} done", end="\r")

        if stopped:
            break

        if block_nr < len(block_order):
            input("\nBlock finished. Take a short break, then press Enter for the next block...")
            # A break is wanted here anyway: the adaptation from this block has to wear off before we start adapting to the opposite end of the scale.
            show(handle, blank, 30.0)

finally:
    if screen is not None:
        plt.close(screen)

    # Saved inside 'finally' so the data survives even if the experiment was cut short.
    out_path = Path(f"experiment_3_{STUDY_NR}.csv")
    with out_path.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["adapting_image", "test_image"] + [f"rating_{i}" for i in range(1, REPEATS + 1)])
        for adaptor in ADAPTORS:
            for test in TESTS:
                w.writerow([adaptor, test] + results[(adaptor, test)])

    n_done = sum(len(v) for v in results.values())
    print(f"\nSaved {n_done} ratings to {out_path}")

INSTRUCTIONS FOR THE PARTICIPANT
- Keep your eyes on the cross in the middle of the image at all times.
- A face will be shown for a long time. Keep looking at the cross, do not look around it.
- Then a face flashes up very briefly. Rate that face, 1 = male and 5 = female.
- Answer with your first impression. It is supposed to be hard.
- Type your answer in the notebook, then look back at the cross straight away.




--- Block 1 of 2 ---
  trial 15/15 done
--- Block 2 of 2 ---
  trial 15/15 done
Saved 30 ratings to experiment_3_s224184.csv
